# Evaluating Supervised and Unsupervised Land-Cover Classification in Urban Environments  
### A Sentinel-2 Case Study of Enfield, London

This notebook compares **K-means clustering** and **Random Forest classification** for four-class land-cover mapping in Enfield, using a 2021 Sentinel-2 surface-reflectance composite and a remapped ESA WorldCover reference layer.

## Project Motivation and Research Question

Urban land-cover classification is challenging because built surfaces, vegetation, water, and open ground often occur in close proximity and can overlap spectrally within the same Sentinel-2 neighbourhood. This is especially relevant in Enfield, where residential areas, parks, reservoirs, transport corridors, and sparsely vegetated surfaces are all present within a single borough-scale study area.

The analysis compares an **unsupervised spectral baseline** with a **supervised label-driven classifier** using the same feature stack and the same four target classes:

- **Built-up / Urban**
- **Vegetation**
- **Water**
- **Open land / Bare or sparsely vegetated surfaces**

**Research question:**  
How do supervised and unsupervised methods differ in their representation of borough-scale urban land cover in Enfield, and what does this reveal about interpretability, reference-label agreement, and computational cost?

> Run the notebook top-to-bottom in Google Colab. If Earth Engine authentication fails, set `USE_EE = False`, provide the two fallback GeoTIFF paths, and continue from the pre-processing section.

In [56]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)


Mounted at /content/drive


In [57]:
from pathlib import Path

PROJECT_DIR = Path("/content/GEOL0069-Enfield-LandCover-Classification")
DATA_DIR = PROJECT_DIR / "data"
FIGURE_DIR = PROJECT_DIR / "figures"

DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project folder: {PROJECT_DIR}")
print(f"Data folder: {DATA_DIR}")
print(f"Figure folder: {FIGURE_DIR}")


Project folder: /content/GEOL0069-Enfield-LandCover-Classification
Data folder: /content/GEOL0069-Enfield-LandCover-Classification/data
Figure folder: /content/GEOL0069-Enfield-LandCover-Classification/figures


In [58]:
from shutil import copy2

DRIVE_DATA_DIR = Path("/content/drive/MyDrive/GEOL0069")

data_files = [
    "enfield_boundary_gla.geojson",
    "enfield_s2_features_2021_20m.tif",
    "enfield_worldcover4_2021_20m.tif",
]

for filename in data_files:
    src = DRIVE_DATA_DIR / filename
    dst = DATA_DIR / filename

    if not src.exists():
        raise FileNotFoundError(f"Missing source file: {src}")

    copy2(src, dst)
    print(f"Copied: {filename}")

print("Data folder contents:")
for path in sorted(DATA_DIR.iterdir()):
    print(f"- {path.name}")


Copied: enfield_boundary_gla.geojson
Copied: enfield_s2_features_2021_20m.tif
Copied: enfield_worldcover4_2021_20m.tif
Data folder contents:
- enfield_boundary_gla.geojson
- enfield_s2_features_2021_20m.tif
- enfield_worldcover4_2021_20m.tif


In [ ]:
# Install core packages for Earth Engine access, vector handling, and raster processing
!pip -q install earthengine-api geemap geopandas rasterio


In [ ]:
import time
import warnings
from pathlib import Path

import ee
import geemap
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch, FancyArrowPatch
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    silhouette_score,
)
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# Reproducibility
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

# Temporal and spatial settings
START_DATE = "2021-04-01"
END_DATE = "2021-08-31"
SCALE = 20
CLOUD_PROB_THRESHOLD = 40

# Input bands and derived features
RAW_BANDS = ["B2", "B3", "B4", "B8", "B11", "B12"]
FEATURE_NAMES = ["B2", "B3", "B4", "B8", "B11", "B12", "NDVI", "NDBI"]

# Target classes
CLASS_NAMES = {
    1: "Built-up / Urban",
    2: "Vegetation",
    3: "Water",
    4: "Open land / Bare",
}

CLASS_COLORS = {
    1: "#d73027",
    2: "#1a9850",
    3: "#2b83ba",
    4: "#fdae61",
}

CLASS_VALUES = list(CLASS_NAMES.keys())

# Runtime tracking
timing = {}

## Utilities and helper functions

The helper functions below keep plotting, area accounting, and basic checks consistent across the notebook.

In [ ]:
def initialize_earth_engine():
    """Initialise Earth Engine in Colab, with authentication fallback if needed."""
    try:
        ee.Initialize()
        print("Earth Engine initialized.")
    except Exception:
        print("Earth Engine is not initialized. Starting authentication...")
        try:
            ee.Authenticate()
            ee.Initialize()
            print("Earth Engine initialized after authentication.")
        except Exception as auth_error:
            raise RuntimeError(
                "Earth Engine authentication failed. Set USE_EE = False and provide local GeoTIFF fallback files."
            ) from auth_error


def stretch_rgb(rgb_array, p_low=2, p_high=98):
    """Apply percentile stretch for RGB visualisation."""
    rgb = rgb_array.astype("float32").copy()
    out = np.zeros_like(rgb, dtype="float32")

    for i in range(3):
        band = rgb[:, :, i]
        lo, hi = np.nanpercentile(band, [p_low, p_high])
        out[:, :, i] = 0 if hi <= lo else np.clip((band - lo) / (hi - lo), 0, 1)

    return out


def plot_class_array(class_array, title, ax=None, extent=None):
    """Plot a four-class land-cover map with a consistent legend."""
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 6))

    masked = np.ma.masked_where(~np.isin(class_array, CLASS_VALUES), class_array)
    cmap = ListedColormap([CLASS_COLORS[i] for i in CLASS_VALUES])
    norm = BoundaryNorm([0.5, 1.5, 2.5, 3.5, 4.5], cmap.N)

    ax.imshow(masked, cmap=cmap, norm=norm, extent=extent, origin="upper")
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])

    legend_items = [
        Patch(facecolor=CLASS_COLORS[i], edgecolor="black", label=CLASS_NAMES[i])
        for i in CLASS_VALUES
    ]
    ax.legend(handles=legend_items, loc="lower left", fontsize=8, frameon=True)
    return ax


def area_table_from_map(class_array, method_name, scale=SCALE):
    """Estimate class areas from raster pixel counts."""
    pixel_area_km2 = (scale * scale) / 1_000_000
    valid_pixels = np.isin(class_array, CLASS_VALUES)
    total_area = valid_pixels.sum() * pixel_area_km2

    rows = []
    for class_id in CLASS_VALUES:
        area_km2 = (class_array == class_id).sum() * pixel_area_km2
        rows.append(
            {
                "method": method_name,
                "class_id": class_id,
                "class_name": CLASS_NAMES[class_id],
                "area_km2": area_km2,
                "area_percent": 100 * area_km2 / total_area if total_area > 0 else np.nan,
            }
        )

    return pd.DataFrame(rows)


def require_non_empty(name, array):
    """Raise an error if an expected array is empty."""
    if len(array) == 0:
        raise ValueError(f"{name} is empty. Check the AOI, date range, masks, and input data.")


def add_north_arrow(ax, x=0.94, y=0.92):
    """Add a simple north arrow using axis-relative coordinates."""
    ax.annotate(
        "N",
        xy=(x, y),
        xytext=(x, y - 0.10),
        xycoords="axes fraction",
        ha="center",
        va="center",
        fontsize=10,
        fontweight="bold",
        arrowprops=dict(arrowstyle="-|>", color="black", linewidth=1.2),
    )


def add_scale_bar(ax, length_km=2, x=0.08, y=0.06):
    """Add a scale bar to maps drawn in projected metre-based coordinates."""
    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()

    length_m = length_km * 1000
    x0 = xmin + x * (xmax - xmin)
    y0 = ymin + y * (ymax - ymin)

    ax.plot([x0, x0 + length_m], [y0, y0], color="black", linewidth=2)
    ax.text(
        x0 + length_m / 2,
        y0 + 0.015 * (ymax - ymin),
        f"{length_km} km",
        ha="center",
        va="bottom",
        fontsize=8,
        color="black",
    )


# Study Area

The study area is the **London Borough of Enfield**, defined using an official London borough boundary layer.

Enfield was selected because its mixed urban landscape presents a practical land-cover classification challenge. Dense residential areas, parks, reservoirs, river corridors, transport infrastructure, agricultural edges, and sparsely vegetated surfaces occur within the same borough and often lie close together. At Sentinel-2 resolution, this spatial mixture can create spectral overlap between built-up, vegetation, water, and open land classes.

This makes Enfield an appropriate case study for examining classification uncertainty in urban environments. The value of the study is not simply that the borough contains diverse land-cover types, but that this diversity makes class boundaries harder to define and interpret. Comparing K-means and Random Forest is therefore meaningful because the two methods may respond differently to mixed pixels, reference-label limitations, and ambiguous urban surfaces.


In [ ]:
# Main option: use Google Earth Engine
USE_EE = False

# GitHub-ready local fallback inputs
FEATURE_TIF = DATA_DIR / "enfield_s2_features_2021_20m.tif"
LABEL_TIF = DATA_DIR / "enfield_worldcover4_2021_20m.tif"
BOUNDARY_GEOJSON = DATA_DIR / "enfield_boundary_gla.geojson"

In [ ]:
# Retrieve Enfield and Greater London borough boundaries for context mapping.
# The Enfield boundary is read from the project data folder for reproducibility.

enfield_gdf = gpd.read_file(BOUNDARY_GEOJSON).to_crs("EPSG:4326")

if enfield_gdf.empty:
    raise ValueError("The local Enfield boundary file returned no features.")

if "name" not in enfield_gdf.columns:
    enfield_gdf["name"] = "Enfield"

if "gss_code" not in enfield_gdf.columns:
    enfield_gdf["gss_code"] = "E09000010"

enfield_gdf["hectares"] = enfield_gdf.to_crs("EPSG:27700").area / 10_000

# Greater London boroughs for context mapping
london_url = (
    "https://services2.arcgis.com/HGokIRbN2kiuIxW5/ArcGIS/rest/services/"
    "London_Borough_Boundaries/FeatureServer/0/query"
    "?where=1%3D1&outFields=*&returnGeometry=true&outSR=4326&f=geojson"
)

london_response = requests.get(london_url, timeout=60)
london_response.raise_for_status()
london_geojson = london_response.json()

london_gdf = gpd.GeoDataFrame.from_features(london_geojson["features"], crs="EPSG:4326")

if london_gdf.empty:
    raise ValueError("The London borough boundary query returned no features.")

if "name" not in london_gdf.columns:
    london_gdf["name"] = london_gdf["NAME"]

display(enfield_gdf[["name", "gss_code", "hectares"]].round(3))



In [ ]:
london_plot_gdf = london_gdf.to_crs("EPSG:27700")
enfield_plot_gdf = enfield_gdf.to_crs("EPSG:27700")

fig, axes = plt.subplots(1, 2, figsize=(10, 4.8))

# Context map: Enfield within Greater London
london_plot_gdf.plot(ax=axes[0], facecolor="#eeeeee", edgecolor="#999999", linewidth=0.4)
enfield_plot_gdf.plot(ax=axes[0], facecolor="#c6dbef", edgecolor="black", linewidth=0.9)
axes[0].set_title("(a) Enfield within Greater London")
axes[0].set_axis_off()
add_north_arrow(axes[0], x=0.92, y=0.88)
add_scale_bar(axes[0], length_km=10, x=0.08, y=0.07)

# Study boundary map
enfield_plot_gdf.plot(ax=axes[1], facecolor="#f2f2f2", edgecolor="black", linewidth=1.2)
axes[1].set_title("(b) Enfield study boundary")
axes[1].set_axis_off()
add_north_arrow(axes[1], x=0.16, y=0.30)
add_scale_bar(axes[1], length_km=2, x=0.08, y=0.07)

plt.tight_layout()
fig.savefig(FIGURE_DIR / "study_area.png", dpi=300, bbox_inches="tight")
plt.show()







# Data Source and Pre-processing

This stage prepares the shared inputs used by both classification methods. It combines the Enfield boundary, Sentinel-2 Level-2A surface reflectance, Sentinel-2 cloud probability, and ESA WorldCover 2021 labels to produce an 8-feature Sentinel-2 stack and a four-class reference label array.

The analysis uses **April-August 2021** imagery. This leaf-on period was selected because vegetation is more spectrally distinct from built-up and bare surfaces, while avoiding unnecessary seasonal variation from winter conditions. A broad scene-level cloud filter (`CLOUDY_PIXEL_PERCENTAGE < 80`) is used to retain enough summer observations, while pixel-level cloud masking is applied more strictly using Sentinel-2 cloud probability (`CLOUD_PROB_THRESHOLD = 40`), consistent with the commonly used **30-50%** range for cloud-probability masking.

A **20 m** analysis scale was chosen as a practical compromise between spatial detail and Colab runtime stability. In the local GeoTIFF fallback workflow, the feature and label rasters are checked after loading to confirm that they use the same projected coordinate reference system and pixel size. The expected CRS is **EPSG:32630 (UTM Zone 30N)**, whose map units are metres; therefore, class-area estimates treat each 20 m pixel as **400 m²** before converting totals to km².

The ESA WorldCover classes are remapped to the four target classes shown below:

| WorldCover class | Target class |
|---|---|
| Tree cover, shrubland, grassland, cropland, herbaceous wetland | Vegetation |
| Built-up | Built-up / Urban |
| Permanent water bodies | Water |
| Bare / sparse vegetation, moss and lichen | Open land / Bare |
| Snow and ice, mangroves | Ignored |



In [ ]:
t0 = time.perf_counter()

import json

if "enfield_gdf" not in globals():
    raise NameError("Run the Study Area boundary cell first so enfield_gdf is available.")

enfield_gdf_wgs84 = enfield_gdf.to_crs("EPSG:4326").copy()
enfield_geojson = json.loads(enfield_gdf_wgs84.to_json())


def find_existing_file(candidates):
    checked = []
    for item in candidates:
        path = Path(item).expanduser()
        checked.append(str(path))
        if path.exists():
            return path, checked
    return None, checked


use_local_fallback = not USE_EE
data_source = None
raster_crs = None
raster_res = None

if USE_EE:
    try:
        initialize_earth_engine()
        aoi = geemap.geojson_to_ee(enfield_geojson).geometry()

        s2_sr = (
            ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
            .filterBounds(aoi)
            .filterDate(START_DATE, END_DATE)
            .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 80))
        )

        s2_clouds = (
            ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY")
            .filterBounds(aoi)
            .filterDate(START_DATE, END_DATE)
        )

        join = ee.Join.saveFirst("cloud_probability")
        joined = ee.ImageCollection(
            join.apply(
                primary=s2_sr,
                secondary=s2_clouds,
                condition=ee.Filter.equals(leftField="system:index", rightField="system:index"),
            )
        ).filter(ee.Filter.notNull(["cloud_probability"]))

        s2_count = joined.size().getInfo()
        if s2_count == 0:
            raise ValueError(
                f"No joined Sentinel-2/cloud-probability scenes found for {START_DATE} to {END_DATE}."
            )

        print(f"Sentinel-2 scenes after filtering: {s2_count}")

        def mask_s2_clouds_and_shadows(image):
            cloud_prob = ee.Image(image.get("cloud_probability")).select("probability")
            scl = image.select("SCL")

            cloud_mask = cloud_prob.lt(CLOUD_PROB_THRESHOLD)
            scl_mask = (
                scl.neq(0)
                .And(scl.neq(1))
                .And(scl.neq(3))
                .And(scl.neq(8))
                .And(scl.neq(9))
                .And(scl.neq(10))
                .And(scl.neq(11))
            )

            return image.select(RAW_BANDS).multiply(0.0001).updateMask(cloud_mask.And(scl_mask))

        s2_composite = joined.map(mask_s2_clouds_and_shadows).median().clip(aoi)
        ndvi = s2_composite.normalizedDifference(["B8", "B4"]).rename("NDVI")
        ndbi = s2_composite.normalizedDifference(["B11", "B8"]).rename("NDBI")

        feature_image = (
            s2_composite
            .addBands([ndvi, ndbi])
            .select(FEATURE_NAMES)
            .toFloat()
            .clip(aoi)
        )

        worldcover = ee.ImageCollection("ESA/WorldCover/v200").first().select("Map").clip(aoi)
        label4 = worldcover.remap(
            [10, 20, 30, 40, 50, 60, 80, 90, 100],
            [2, 2, 2, 2, 1, 4, 3, 2, 4],
            0,
        ).rename("label").toByte().clip(aoi)

        feature_arr = geemap.ee_to_numpy(feature_image, region=aoi, scale=SCALE)
        label_arr = geemap.ee_to_numpy(label4, region=aoi, scale=SCALE)

        if feature_arr is None or label_arr is None:
            raise RuntimeError("Earth Engine returned empty arrays.")

        enfield_gdf = enfield_gdf_wgs84.copy()
        extent = list(enfield_gdf.total_bounds)
        data_source = "Earth Engine"

    except Exception as ee_error:
        print("Earth Engine route failed:")
        print(f"{type(ee_error).__name__}: {ee_error}")
        print("Switching to local GeoTIFF fallback.")
        use_local_fallback = True

if use_local_fallback:
    import rasterio

    feature_candidates = [
        FEATURE_TIF,
        "GEOL0069/enfield_s2_features_2021_20m.tif",
        Path.cwd() / "GEOL0069" / "enfield_s2_features_2021_20m.tif",
        "/content/GEOL0069/enfield_s2_features_2021_20m.tif",
        "/content/drive/MyDrive/GEOL0069/enfield_s2_features_2021_20m.tif",
        "/content/drive/MyDrive/Project/GEOL0069/enfield_s2_features_2021_20m.tif",
    ]

    label_candidates = [
        LABEL_TIF,
        "GEOL0069/enfield_worldcover4_2021_20m.tif",
        Path.cwd() / "GEOL0069" / "enfield_worldcover4_2021_20m.tif",
        "/content/GEOL0069/enfield_worldcover4_2021_20m.tif",
        "/content/drive/MyDrive/GEOL0069/enfield_worldcover4_2021_20m.tif",
        "/content/drive/MyDrive/Project/GEOL0069/enfield_worldcover4_2021_20m.tif",
    ]

    feature_path, checked_feature = find_existing_file(feature_candidates)
    label_path, checked_label = find_existing_file(label_candidates)

    if feature_path is None:
        raise FileNotFoundError(
            "Feature GeoTIFF not found.\n"
            "Checked:\n- " + "\n- ".join(checked_feature)
        )

    if label_path is None:
        raise FileNotFoundError(
            "Label GeoTIFF not found.\n"
            "Checked:\n- " + "\n- ".join(checked_label)
        )

    with rasterio.open(feature_path) as src:
        feature_arr = np.moveaxis(src.read(), 0, -1)
        raster_crs = src.crs
        raster_bounds = src.bounds
        raster_res = src.res

    with rasterio.open(label_path) as src:
        label_arr = src.read(1)
        label_crs = src.crs
        label_res = src.res

    if raster_crs != label_crs:
        raise ValueError(f"Feature and label CRS do not match: {raster_crs} vs {label_crs}")

    if not raster_crs.is_projected:
        raise ValueError(
            f"Raster CRS is not projected: {raster_crs}. "
            "Area estimates require metre-based projected coordinates."
        )

    if not np.isclose(abs(raster_res[0]), SCALE) or not np.isclose(abs(raster_res[1]), SCALE):
        raise ValueError(
            f"Feature raster pixel size {raster_res} does not match expected SCALE={SCALE} m."
        )

    if not np.isclose(abs(label_res[0]), SCALE) or not np.isclose(abs(label_res[1]), SCALE):
        raise ValueError(
            f"Label raster pixel size {label_res} does not match expected SCALE={SCALE} m."
        )

    enfield_gdf = enfield_gdf_wgs84.to_crs(raster_crs)
    extent = [raster_bounds.left, raster_bounds.bottom, raster_bounds.right, raster_bounds.top]
    data_source = f"Local GeoTIFF fallback ({feature_path.name})"

feature_arr = np.asarray(feature_arr, dtype="float32")
label_arr = np.squeeze(np.asarray(label_arr))

if feature_arr.ndim != 3:
    raise ValueError(f"feature_arr should be 3D, got shape {feature_arr.shape}.")

if label_arr.ndim != 2:
    raise ValueError(f"label_arr should be 2D, got shape {label_arr.shape}.")

if feature_arr.shape[:2] != label_arr.shape:
    raise ValueError(
        f"Feature and label grids do not match: {feature_arr.shape[:2]} vs {label_arr.shape}."
    )

label_arr = np.where(np.isfinite(label_arr), label_arr, 0).astype("uint8")

valid_mask = np.isfinite(feature_arr).all(axis=2)
valid_mask &= np.nanmean(np.abs(feature_arr[:, :, :6]), axis=2) > 0
labelled_mask = valid_mask & np.isin(label_arr, CLASS_VALUES)

X_all = feature_arr[valid_mask]
X_labelled = feature_arr[labelled_mask]
y_labelled = label_arr[labelled_mask].astype("uint8")
labelled_rows, labelled_cols = np.where(labelled_mask)

require_non_empty("Valid Sentinel-2 feature pixels", X_all)
require_non_empty("WorldCover-labelled pixels", X_labelled)

timing["preprocessing"] = time.perf_counter() - t0

label_counts = (
    pd.Series(y_labelled)
    .map(CLASS_NAMES)
    .value_counts()
    .reindex(CLASS_NAMES.values(), fill_value=0)
)

print("Pre-processing summary")
print(f"- Data source: {data_source}")

if raster_crs is not None:
    print(f"- Raster CRS: {raster_crs}")
    print(f"- Raster pixel size: {abs(raster_res[0]):.1f} m x {abs(raster_res[1]):.1f} m")

print(f"- Feature array shape: {feature_arr.shape}")
print(f"- Label array shape: {label_arr.shape}")
print(f"- Valid pixel count: {X_all.shape[0]:,}")
print(f"- Labelled pixel count: {X_labelled.shape[0]:,}")
print(f"- Runtime: {timing['preprocessing']:.1f} seconds")

print("\nLabelled pixel count by class")
for class_name, count in label_counts.items():
    print(f"- {class_name}: {count:,}")





In [ ]:
rgb = feature_arr[:, :, [2, 1, 0]].copy()
rgb[~valid_mask] = np.nan
rgb_display = stretch_rgb(rgb)

fig, ax = plt.subplots(figsize=(6.2, 6))
ax.imshow(
    rgb_display,
    extent=[extent[0], extent[2], extent[1], extent[3]],
    origin="upper",
)
enfield_gdf.boundary.plot(ax=ax, color="black", linewidth=0.8)

ax.set_title("Sentinel-2 RGB composite (April-August 2021)")
ax.set_xticks([])
ax.set_yticks([])

add_north_arrow(ax, x=0.16, y=0.25)
add_scale_bar(ax, length_km=2, x=0.08, y=0.07)

plt.tight_layout()
fig.savefig(FIGURE_DIR / "sentinel2_rgb.png", dpi=300, bbox_inches="tight")
plt.show()






# Method Overview

Both classification methods use the same 2021 Sentinel-2 feature stack, so differences in the outputs reflect the modelling approach rather than differences in input data. The project compares both methods under the same four target land-cover classes, making the outputs directly comparable at borough scale.

## K-means workflow

K-means is used as an unsupervised spectral baseline. Because the project compares both methods under a shared four-class scheme, the final K-means model uses `k = 4` to match the target comparison framework. Elbow and silhouette diagnostics are still reported as a sensitivity check, rather than as a purely automatic model-selection procedure.

The model is fitted to a scaled sample of valid pixels from the Enfield composite. A maximum of 50,000 valid pixels is sampled to keep the clustering step computationally manageable while still using a substantial share of the borough-wide valid pixels; the sampled proportion is printed below for transparency. The selected clustering is then interpreted using cluster feature profiles and map inspection before being translated into the common four-class scheme.



In [ ]:
fig, ax = plt.subplots(figsize=(9.2, 2.3))
ax.axis("off")

steps = [
    "Feature stack",
    "Scale and sample",
    "Select k",
    "Fit K-means",
    "Interpret clusters",
    "Four-class map",
]

x_positions = np.linspace(0.06, 0.94, len(steps))
for i, (x, label) in enumerate(zip(x_positions, steps)):
    ax.text(
        x,
        0.5,
        label,
        ha="center",
        va="center",
        fontsize=9,
        bbox=dict(
            boxstyle="round,pad=0.32",
            facecolor="#f4f4f4",
            edgecolor="#666666",
            linewidth=0.9,
        ),
    )
    if i < len(steps) - 1:
        ax.add_patch(
            FancyArrowPatch(
                (x + 0.06, 0.5),
                (x_positions[i + 1] - 0.06, 0.5),
                arrowstyle="->",
                mutation_scale=10,
                linewidth=1,
                color="#666666",
            )
        )

ax.set_title("K-means workflow", pad=8)
fig.savefig(FIGURE_DIR / "kmeans_workflow.png", dpi=300, bbox_inches="tight")
plt.show()



In [ ]:
t0 = time.perf_counter()

require_non_empty("Valid Sentinel-2 feature pixels", X_all)

KMEANS_SAMPLE_SIZE = min(50_000, X_all.shape[0])
kmeans_sample_percent = 100 * KMEANS_SAMPLE_SIZE / X_all.shape[0]

print(
    f"K-means sample: {KMEANS_SAMPLE_SIZE:,} pixels "
    f"({kmeans_sample_percent:.1f}% of valid pixels)"
)

if KMEANS_SAMPLE_SIZE < 1000:
    print("Warning: K-means is running with a small sample; cluster stability may be weak.")

sample_indices = rng.choice(X_all.shape[0], size=KMEANS_SAMPLE_SIZE, replace=False)
X_kmeans_sample = X_all[sample_indices]

scaler_kmeans = StandardScaler()
X_kmeans_scaled = scaler_kmeans.fit_transform(X_kmeans_sample)

candidate_k = [3, 4, 5, 6]
k_metrics = []

silhouette_n = min(10_000, X_kmeans_scaled.shape[0])
silhouette_indices = rng.choice(X_kmeans_scaled.shape[0], size=silhouette_n, replace=False)
X_silhouette = X_kmeans_scaled[silhouette_indices]

for k in candidate_k:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels_full = km.fit_predict(X_kmeans_scaled)
    labels_silhouette = labels_full[silhouette_indices]
    sil = silhouette_score(X_silhouette, labels_silhouette)
    k_metrics.append(
        {
            "k": k,
            "inertia": km.inertia_,
            "silhouette": sil,
        }
    )

k_metrics_df = pd.DataFrame(k_metrics)
display(k_metrics_df.round(3))

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))

axes[0].plot(k_metrics_df["k"], k_metrics_df["inertia"], marker="o", color="#4c78a8")
axes[0].set_title("Elbow diagnostic")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Inertia")

axes[1].plot(k_metrics_df["k"], k_metrics_df["silhouette"], marker="o", color="#4c78a8")
axes[1].set_title("Silhouette diagnostic")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Score")

plt.tight_layout()
fig.savefig(FIGURE_DIR / "kmeans_diagnostics.png", dpi=300, bbox_inches="tight")
plt.show()


selected_k = 4
print(
    f"Selected k for final model: {selected_k} "
    "(pre-specified to match the four target land-cover classes; "
    "elbow and silhouette diagnostics are reported as sensitivity checks, not as automatic selection criteria)."
)

kmeans = KMeans(n_clusters=selected_k, random_state=RANDOM_STATE, n_init=10)
kmeans.fit(X_kmeans_scaled)

cluster_pred = kmeans.predict(scaler_kmeans.transform(X_all))
cluster_map = np.full(valid_mask.shape, -1, dtype="int16")
cluster_map[valid_mask] = cluster_pred

timing["kmeans"] = time.perf_counter() - t0
print(f"K-means runtime: {timing['kmeans']:.1f} seconds")


## Heuristic cluster interpretation

The outputs below summarise the spectral behaviour of the four clusters before they are translated into the common comparison scheme. These suggested class labels are based on cluster feature profiles and map inspection, so they should be read as heuristic interpretation rather than validated land-cover classes. In particular, the Built-up / Urban cluster represents mixed urban fabric at 20 m resolution rather than pure impervious surface.



In [ ]:
cluster_summary = []
total_valid_pixels = X_all.shape[0]

for cluster_id in sorted(np.unique(cluster_pred)):
    pixels = X_all[cluster_pred == cluster_id]
    means = pixels.mean(axis=0)

    row = {
        "cluster": int(cluster_id),
        "pixel_count": int(pixels.shape[0]),
        "pixel_percent": 100 * pixels.shape[0] / total_valid_pixels,
    }
    row.update({band: means[i] for i, band in enumerate(FEATURE_NAMES)})
    cluster_summary.append(row)

cluster_summary_df = pd.DataFrame(cluster_summary)

display(
    cluster_summary_df[
        ["cluster", "pixel_count", "pixel_percent", "NDVI", "NDBI", "B8", "B11"]
    ]
    .sort_values("cluster")
    .round(3)
)

centroid_plot_df = cluster_summary_df.set_index("cluster")[FEATURE_NAMES]

fig, ax = plt.subplots(figsize=(8.2, 4.2))
for cluster_id, row in centroid_plot_df.iterrows():
    ax.plot(FEATURE_NAMES, row.values, marker="o", linewidth=1.8, label=f"Cluster {cluster_id}")

ax.set_title("K-means cluster mean feature profiles")
ax.set_ylabel("Mean scaled feature value")
ax.tick_params(axis="x", rotation=35)
ax.legend(ncol=2, fontsize=8, frameon=True)
plt.tight_layout()
fig.savefig(FIGURE_DIR / "kmeans_cluster_profiles.png", dpi=300, bbox_inches="tight")
plt.show()



In [ ]:
# Post-hoc cluster interpretation for the common four-class comparison scheme.
# This mapping is based on cluster feature profiles and map inspection, not on validated ground truth.
cluster_to_class = {
    0: 3,  # Water
    1: 2,  # Vegetation
    2: 1,  # Built-up / Urban
    3: 4,  # Open land / Bare
}

cluster_rationale_map = {
    0: "Lowest NDVI and lowest NIR response, consistent with water bodies.",
    1: "Highest NDVI and NIR response, consistent with vegetated surfaces.",
    2: (
        "Moderate NDVI suggests mixed residential urban pixels rather than pure built-up surface; "
        "mapped as Built-up / Urban because it represents the dominant urban fabric in the four-class comparison."
    ),
    3: (
        "Low vegetation signal with relatively stronger SWIR response, consistent with open land, "
        "bare ground, and sparsely vegetated surfaces."
    ),
}

cluster_interpretation_df = cluster_summary_df[
    ["cluster", "pixel_count", "pixel_percent", "NDVI", "NDBI", "B8", "B11"]
].copy()

cluster_interpretation_df["suggested_class_id"] = cluster_interpretation_df["cluster"].map(cluster_to_class)
cluster_interpretation_df["suggested_class"] = cluster_interpretation_df["suggested_class_id"].map(CLASS_NAMES)
cluster_interpretation_df["short_rationale"] = cluster_interpretation_df["cluster"].map(cluster_rationale_map)

display(
    cluster_interpretation_df.rename(
        columns={
            "NDVI": "mean_NDVI",
            "NDBI": "mean_NDBI",
            "pixel_percent": "pixel_percent",
        }
    )[
        [
            "cluster",
            "pixel_count",
            "pixel_percent",
            "mean_NDVI",
            "mean_NDBI",
            "suggested_class",
            "short_rationale",
        ]
    ].round(3)
)

kmeans_class_map = np.zeros(valid_mask.shape, dtype="uint8")
for cluster_id, class_id in cluster_to_class.items():
    kmeans_class_map[cluster_map == cluster_id] = class_id

kmeans_area_df = area_table_from_map(kmeans_class_map, "K-means")

kmeans_total_row = pd.DataFrame(
    [
        {
            "method": "K-means",
            "class_id": "Total",
            "class_name": "Total",
            "area_km2": kmeans_area_df["area_km2"].sum(),
            "area_percent": kmeans_area_df["area_percent"].sum(),
        }
    ]
)

kmeans_area_display_df = pd.concat([kmeans_area_df, kmeans_total_row], ignore_index=True)
display(kmeans_area_display_df.round(2))


## Random Forest workflow

Random Forest uses the same feature stack together with the remapped reference raster. It is trained and evaluated with a simple spatial hold-out split before being applied to all valid Enfield pixels.

In [ ]:
fig, ax = plt.subplots(figsize=(9.2, 2.3))
ax.axis("off")

steps = [
    "Feature stack",
    "Remap labels",
    "Spatial split",
    "Train RF",
    "Evaluate",
    "Four-class map",
]

x_positions = np.linspace(0.06, 0.94, len(steps))
for i, (x, label) in enumerate(zip(x_positions, steps)):
    ax.text(
        x,
        0.5,
        label,
        ha="center",
        va="center",
        fontsize=9,
        bbox=dict(
            boxstyle="round,pad=0.32",
            facecolor="#f4f4f4",
            edgecolor="#666666",
            linewidth=0.9,
        ),
    )
    if i < len(steps) - 1:
        ax.add_patch(
            FancyArrowPatch(
                (x + 0.06, 0.5),
                (x_positions[i + 1] - 0.06, 0.5),
                arrowstyle="->",
                mutation_scale=10,
                linewidth=1,
                color="#666666",
            )
        )

ax.set_title("Random Forest workflow", pad=8)
fig.savefig(FIGURE_DIR / "rf_workflow.png", dpi=300, bbox_inches="tight")
plt.show()




In [ ]:
t0 = time.perf_counter()

RF_TRAIN_CAP_PER_CLASS = 2500
RF_TEST_CAP_PER_CLASS = 1000
MIN_SAMPLES_PER_CLASS = 15

height, width = label_arr.shape
candidate_grids = [(4, 4), (3, 3), (5, 5), (4, 3), (3, 4), (2, 2)]

split_found = False
best_train_pool_mask = None
best_test_pool_mask = None
split_method = None

for n_block_rows, n_block_cols in candidate_grids:
    block_rows = np.minimum((labelled_rows / height * n_block_rows).astype(int), n_block_rows - 1)
    block_cols = np.minimum((labelled_cols / width * n_block_cols).astype(int), n_block_cols - 1)
    block_id = block_rows * n_block_cols + block_cols

    for divisor in [4, 3, 5, 2]:
        for offset in range(divisor):
            candidate_test = (block_id % divisor) == offset
            candidate_train = ~candidate_test
            test_fraction = candidate_test.mean()

            ok_fraction = 0.15 <= test_fraction <= 0.40
            ok_classes = all(
                ((y_labelled[candidate_train] == class_id).sum() >= MIN_SAMPLES_PER_CLASS)
                and ((y_labelled[candidate_test] == class_id).sum() >= MIN_SAMPLES_PER_CLASS)
                for class_id in CLASS_VALUES
            )

            if ok_fraction and ok_classes:
                best_train_pool_mask = candidate_train
                best_test_pool_mask = candidate_test
                split_method = (
                    f"Block-based spatial hold-out ({n_block_rows}x{n_block_cols} grid, divisor={divisor})"
                )
                split_found = True
                break
        if split_found:
            break
    if split_found:
        break

if split_found:
    print(f"Using {split_method}.")
    train_pool_mask = best_train_pool_mask
    test_pool_mask = best_test_pool_mask
else:
    split_method = "Stratified random pixel fallback"
    print(
        "Warning: no valid spatial split preserved all classes. "
        "Using a stratified random pixel split instead, which is less conservative because of spatial autocorrelation."
    )

    random_test = np.zeros(len(y_labelled), dtype=bool)
    for class_id in CLASS_VALUES:
        class_idx = np.flatnonzero(y_labelled == class_id)

        if len(class_idx) < 2 * MIN_SAMPLES_PER_CLASS:
            raise ValueError(
                f"Class '{CLASS_NAMES[class_id]}' has too few labelled pixels for train/test splitting."
            )

        test_n = max(MIN_SAMPLES_PER_CLASS, int(0.25 * len(class_idx)))
        test_n = min(test_n, len(class_idx) - MIN_SAMPLES_PER_CLASS)

        chosen_test = rng.choice(class_idx, size=test_n, replace=False)
        random_test[chosen_test] = True

    train_pool_mask = ~random_test
    test_pool_mask = random_test

spatial_split_map = np.zeros(label_arr.shape, dtype="uint8")
spatial_split_map[labelled_rows[train_pool_mask], labelled_cols[train_pool_mask]] = 1
spatial_split_map[labelled_rows[test_pool_mask], labelled_cols[test_pool_mask]] = 2

train_indices_all = np.flatnonzero(train_pool_mask)
test_indices_all = np.flatnonzero(test_pool_mask)

sample_rows = []
train_indices = []
test_indices = []

for class_id in CLASS_VALUES:
    train_class_indices = train_indices_all[y_labelled[train_indices_all] == class_id]
    test_class_indices = test_indices_all[y_labelled[test_indices_all] == class_id]

    if len(train_class_indices) < MIN_SAMPLES_PER_CLASS or len(test_class_indices) < MIN_SAMPLES_PER_CLASS:
        raise ValueError(
            f"Class '{CLASS_NAMES[class_id]}' has too few samples after splitting "
            f"(train={len(train_class_indices)}, test={len(test_class_indices)})."
        )

    train_n = min(RF_TRAIN_CAP_PER_CLASS, len(train_class_indices))
    test_n = min(RF_TEST_CAP_PER_CLASS, len(test_class_indices))

    if train_n < RF_TRAIN_CAP_PER_CLASS:
        print(f"Training sample for {CLASS_NAMES[class_id]} limited to {train_n}.")
    if test_n < RF_TEST_CAP_PER_CLASS:
        print(f"Test sample for {CLASS_NAMES[class_id]} limited to {test_n}.")

    chosen_train = rng.choice(train_class_indices, size=train_n, replace=False)
    chosen_test = rng.choice(test_class_indices, size=test_n, replace=False)

    train_indices.append(chosen_train)
    test_indices.append(chosen_test)

    sample_rows.append(
        {
            "class_id": class_id,
            "class_name": CLASS_NAMES[class_id],
            "train_available": len(train_class_indices),
            "train_sampled": train_n,
            "test_available": len(test_class_indices),
            "test_sampled": test_n,
        }
    )

sampling_df = pd.DataFrame(sample_rows)
display(sampling_df)

train_indices = np.concatenate(train_indices)
test_indices = np.concatenate(test_indices)

X_train = X_labelled[train_indices]
y_train = y_labelled[train_indices]
X_test = X_labelled[test_indices]
y_test = y_labelled[test_indices]

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_STATE,
    class_weight="balanced",
    n_jobs=-1,
    max_features="sqrt",
)

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

overall_accuracy = accuracy_score(y_test, y_pred)
kappa = cohen_kappa_score(y_test, y_pred)

print(f"Split method: {split_method}")
print(f"Overall agreement with WorldCover-derived labels: {overall_accuracy:.3f}")
print(f"Cohen's kappa: {kappa:.3f}")

report_df = pd.DataFrame(
    classification_report(
        y_test,
        y_pred,
        labels=CLASS_VALUES,
        target_names=[CLASS_NAMES[i] for i in CLASS_VALUES],
        output_dict=True,
        zero_division=0,
    )
).T
display(report_df.round(3))

rf_full_pred = rf.predict(X_all)
rf_class_map = np.zeros(valid_mask.shape, dtype="uint8")
rf_class_map[valid_mask] = rf_full_pred.astype("uint8")

rf_area_df = area_table_from_map(rf_class_map, "Random Forest")
importance_df = pd.DataFrame(
    {"feature": FEATURE_NAMES, "importance": rf.feature_importances_}
).sort_values("importance", ascending=True)

timing["random_forest"] = time.perf_counter() - t0
print(f"Random Forest runtime: {timing['random_forest']:.1f} seconds")


# Results and Comparison

This section brings together the final classification maps, the Random Forest evaluation outputs, the spatial hold-out design, and the class-area comparison for Enfield. Together, these results show where the supervised and unsupervised approaches converge, where they diverge, and what this implies for borough-scale urban land-cover classification.

## Final classification maps

The two maps below show the common four-class land-cover scheme derived from the K-means and Random Forest workflows.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 6.0))

plot_class_array(
    kmeans_class_map,
    "K-means four-class comparison map",
    ax=axes[0],
    extent=[extent[0], extent[2], extent[1], extent[3]],
)
enfield_gdf.boundary.plot(ax=axes[0], color="black", linewidth=0.7)
axes[0].get_legend().remove()
add_north_arrow(axes[0], x=0.12, y=0.26)
add_scale_bar(axes[0], length_km=2, x=0.08, y=0.07)

plot_class_array(
    rf_class_map,
    "Random Forest four-class map",
    ax=axes[1],
    extent=[extent[0], extent[2], extent[1], extent[3]],
)
enfield_gdf.boundary.plot(ax=axes[1], color="black", linewidth=0.7)
axes[1].get_legend().remove()
add_north_arrow(axes[1], x=0.12, y=0.26)
add_scale_bar(axes[1], length_km=2, x=0.08, y=0.07)

legend_items = [
    Patch(facecolor=CLASS_COLORS[i], edgecolor="black", label=CLASS_NAMES[i])
    for i in CLASS_VALUES
]

fig.legend(
    handles=legend_items,
    loc="lower center",
    ncol=4,
    fontsize=10,
    frameon=True,
    bbox_to_anchor=(0.5, 0.055),
)

plt.subplots_adjust(left=0.04, right=0.98, top=0.90, bottom=0.13, wspace=0.06)
fig.savefig(FIGURE_DIR / "final_classification_maps.png", dpi=300, bbox_inches="tight")
plt.show()









## RF evaluation

The panel below summarises Random Forest agreement with the WorldCover-derived reference labels and highlights the features that contribute most strongly to the supervised model. The evaluation is based on the spatial hold-out test set, so the reported scores should be interpreted as reference-label agreement rather than independent field-validated accuracy.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.6))

cm = confusion_matrix(y_test, y_pred, labels=CLASS_VALUES)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[CLASS_NAMES[i] for i in CLASS_VALUES],
)
disp.plot(ax=axes[0], cmap="Blues", colorbar=False, xticks_rotation=35)
axes[0].set_title(f"RF spatial hold-out evaluation\nOA = {overall_accuracy:.3f}, kappa = {kappa:.3f}")

axes[1].barh(
    importance_df["feature"],
    importance_df["importance"],
    color="#4c78a8",
    edgecolor="none",
)
axes[1].set_title("Random Forest feature importance")
axes[1].set_xlabel("Importance score")

plt.tight_layout()
fig.savefig(FIGURE_DIR / "rf_evaluation.png", dpi=300, bbox_inches="tight")
plt.show()



## Spatial hold-out design

The split is shown here so the Random Forest evaluation design is visible rather than only described in text. By separating training and test pixels spatially, this step provides a more conservative assessment than a purely random pixel split, which would be more vulnerable to spatial autocorrelation and inflated agreement.


In [ ]:
split_cmap = ListedColormap(["#ffffff", "#4c78a8", "#f58518"])
split_norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5], split_cmap.N)

fig, ax = plt.subplots(figsize=(6.8, 6.4))
masked_split = np.ma.masked_where(spatial_split_map == 0, spatial_split_map)

ax.imshow(
    masked_split,
    cmap=split_cmap,
    norm=split_norm,
    extent=[extent[0], extent[2], extent[1], extent[3]],
    origin="upper",
)
enfield_gdf.boundary.plot(ax=ax, color="black", linewidth=0.8)

ax.set_title("Random Forest spatial hold-out design")
ax.set_xticks([])
ax.set_yticks([])

add_north_arrow(ax, x=0.12, y=0.25)
add_scale_bar(ax, length_km=2, x=0.08, y=0.08)

legend_items = [
    Patch(facecolor="#4c78a8", edgecolor="black", label="Training pixels"),
    Patch(facecolor="#f58518", edgecolor="black", label="Test pixels"),
]

fig.legend(
    handles=legend_items,
    loc="lower center",
    ncol=2,
    fontsize=10,
    frameon=True,
    bbox_to_anchor=(0.5, 0.045),
)

fig.text(
    0.5,
    0.005,
    split_method,
    ha="center",
    va="bottom",
    fontsize=9,
)

plt.subplots_adjust(left=0.05, right=0.98, top=0.90, bottom=0.14)
fig.savefig(FIGURE_DIR / "spatial_holdout.png", dpi=300, bbox_inches="tight")
plt.show()





## Class area comparison

The table below summarises class area percentages for the K-means and Random Forest outputs and highlights the main differences between methods. This comparison provides a compact map-level summary of where the two approaches allocate urban, vegetation, water, and open land differently across Enfield.


In [ ]:
area_comparison_df = pd.concat([kmeans_area_df, rf_area_df], ignore_index=True)

class_order = [CLASS_NAMES[i] for i in CLASS_VALUES]
area_pivot = (
    area_comparison_df.pivot(index="class_name", columns="method", values="area_percent")
    .reindex(class_order)
    .reset_index()
)

area_pivot["Difference (RF - K-means, pp)"] = (
    area_pivot["Random Forest"] - area_pivot["K-means"]
)

max_disagreement_class = area_pivot.loc[
    area_pivot["Difference (RF - K-means, pp)"].abs().idxmax(),
    "class_name",
]

def area_comment(row):
    class_name = row["class_name"]
    diff = row["Difference (RF - K-means, pp)"]
    abs_diff = abs(diff)
    direction = "more" if diff > 0 else "less"

    largest_note = ""
    if class_name == max_disagreement_class:
        largest_note = " This is the largest method difference in the table."

    if class_name == "Water":
        if abs_diff < 2:
            return (
                f"Very similar between methods ({diff:+.1f} pp), "
                f"consistent with clear water separability."
                f"{largest_note}"
            )
        return (
            f"RF maps {direction} water ({diff:+.1f} pp), mainly around reservoir margins "
            f"and mixed shoreline pixels."
            f"{largest_note}"
        )

    if class_name == "Vegetation":
        if abs_diff < 5:
            return (
                f"Broadly similar ({diff:+.1f} pp), reflecting the strong vegetation signal "
                f"in the leaf-on composite."
                f"{largest_note}"
            )
        return (
            f"RF maps {direction} vegetation ({diff:+.1f} pp), likely influenced by gardens, "
            f"parks, and mixed urban-green pixels."
            f"{largest_note}"
        )

    if class_name == "Built-up / Urban":
        if abs_diff < 5:
            return (
                f"Urban share is similar overall ({diff:+.1f} pp), although local boundary "
                f"differences remain."
                f"{largest_note}"
            )
        return (
            f"RF maps {direction} built-up land ({diff:+.1f} pp), reflecting its closer "
            f"alignment with the reference label scheme."
            f"{largest_note}"
        )

    if class_name == "Open land / Bare":
        return (
            f"RF maps {direction} open/bare land ({diff:+.1f} pp), mainly where sparse "
            f"vegetation and urban materials overlap spectrally."
            f"{largest_note}"
        )

    return f"Moderate difference ({diff:+.1f} pp).{largest_note}"

area_pivot["Comment"] = area_pivot.apply(area_comment, axis=1)

results_table = area_pivot.rename(
    columns={
        "class_name": "Class",
        "K-means": "K-means (%)",
        "Random Forest": "Random Forest (%)",
    }
)[
    [
        "Class",
        "K-means (%)",
        "Random Forest (%)",
        "Difference (RF - K-means, pp)",
        "Comment",
    ]
]

display(results_table.round(2))


# Discussion and Limitations

## Where K-means adds value

K-means adds value by exposing spectral structure that is compressed once the analysis is translated into a fixed four-class land-cover scheme. In this project, `k = 4` is used to match the shared comparison framework rather than to claim that Enfield naturally contains only four spectral groups. The elbow and silhouette diagnostics show that this choice is defensible for comparison, but the cluster interpretation remains heuristic.

The cluster profile plot and post-hoc interpretation table show that Enfield contains clearly vegetated and water-dominated surfaces, as well as more ambiguous combinations of built materials, sparse vegetation, and open ground. This is visible in the K-means comparison map, where urban fabric and low-vegetation surfaces are represented in a more fragmented but spectrally sensitive way than in the Random Forest output.

The Built-up / Urban cluster should therefore be read carefully. Its moderate NDVI indicates mixed urban fabric rather than pure impervious surface, which is plausible at 20 m resolution because residential pixels often contain buildings, roads, gardens, street trees, and small green spaces. In this sense, K-means is useful for spectral interpretation, but weaker as a final labelled land-cover map because its class names depend on post-hoc interpretation.

## Where Random Forest is more reliable

Random Forest produces the more submission-ready four-class land-cover map because it is tied to named classes and evaluated against the remapped WorldCover reference layer. In this project, the supervised model achieved an overall agreement of 0.898 and a Cohen's kappa of 0.822 under a block-based spatial hold-out design, suggesting strong reference-label agreement for a borough-scale urban classification task. The confusion matrix also shows that Vegetation and Water are the most reliably identified classes, while the final RF map is spatially cleaner and easier to interpret than the K-means output.

At the same time, this result should be read carefully. The evaluation measures agreement with WorldCover-derived labels rather than field-validated accuracy, so the model is best understood as reproducing a reference classification scheme rather than independently verifying the true land cover of Enfield. Its strength is semantic clarity and consistency, not freedom from label uncertainty.

## Which surfaces remain ambiguous in Enfield

The largest differences between methods occur in the balance between Built-up / Urban, Vegetation, and Open land / Bare. In this reproducible run using `RANDOM_STATE = 42`, the class-area comparison shows that Random Forest maps less Built-up / Urban land than K-means (-8.4 percentage points), more Vegetation (+12.7 percentage points), and much less Open land / Bare (-4.6 percentage points), while Water remains very similar between methods (+0.3 percentage points). These differences are consistent with the final maps, where K-means allocates more area to spectrally mixed low-vegetation and bright-surface classes, whereas Random Forest pulls more of these pixels into the Vegetation class under the reference-label scheme.

This ambiguity is expected in Enfield. Car parks, dry grass, bare soil, rail corridors, bright roofs, paved surfaces, and sparsely vegetated ground can overlap spectrally at 20 m resolution, especially in a heterogeneous urban borough. By contrast, large reservoirs and dense vegetation are more distinct, which helps explain why Water and Vegetation are more stable across the two methods.

The Open land / Bare class should be interpreted most cautiously. In the spatial hold-out split, only 177 training samples and 64 test samples were available for this class, far fewer than for Built-up / Urban and Vegetation. Although `class_weight="balanced"` helps reduce class bias during Random Forest fitting, it cannot fully compensate for limited and spectrally mixed reference samples. The lower F1-score for this class is therefore likely caused by both class imbalance and genuine spectral overlap with dry grass, concrete, bright roofs, bare soil, and sparsely vegetated urban surfaces.

## What this means for borough-scale AI4EO workflows

Taken together, the results show a clear trade-off between spectral flexibility and semantic consistency. K-means is more useful for revealing internal variation and mixed urban surfaces, but it depends on post-hoc interpretation and does not naturally produce a clean thematic map. Random Forest is more effective when the goal is a clearly labelled borough-scale product, but it is constrained by the quality, class balance, and definitions of the reference labels used for training.

The spatial hold-out design improves transparency by making the train-test split visible and more conservative than a purely random pixel split, but it does not remove the uncertainty introduced by mixed pixels and imperfect reference data. The broader lesson from this Enfield case study is that supervised and unsupervised classification should not be treated as interchangeable. At borough scale, they answer slightly different questions: Random Forest is better for producing a stable mapped class scheme, while K-means is better for exposing the spectral complexity that such schemes inevitably simplify.


# Environmental Cost

This section provides a simplified estimate of the computational footprint of the notebook using measured stage runtimes and a small set of transparent assumptions. The aim is not precise life-cycle accounting, but a compact comparison of the relative energy use, carbon emissions, and electricity cost of each workflow stage. This estimate reflects the executed Colab run with local fallback inputs and does not include the upstream cost of generating the GeoTIFF data package.

**Formula**

`Energy (kWh) = runtime_hours × assumed_active_power_kW`

`CO2e (g) = energy_kWh × carbon_intensity_kgCO2e_per_kWh × 1000`

`Estimated cost (£) = energy_kWh × electricity_price_gbp_per_kWh`

**Assumptions**

- Active Colab CPU power: 35 W
- Carbon intensity: 0.233 kg CO2e / kWh
- Electricity price: £0.28 / kWh
- GPU: not used


In [ ]:
runtime_df = pd.DataFrame(
    [
        {"stage": "Pre-processing", "runtime_seconds": timing.get("preprocessing", np.nan)},
        {"stage": "K-means", "runtime_seconds": timing.get("kmeans", np.nan)},
        {"stage": "Random Forest", "runtime_seconds": timing.get("random_forest", np.nan)},
    ]
)

runtime_df["runtime_minutes"] = runtime_df["runtime_seconds"] / 60
runtime_df["runtime_hours"] = runtime_df["runtime_seconds"] / 3600

ASSUMED_POWER_KW = 0.035
CARBON_INTENSITY_KG_PER_KWH = 0.233
ELECTRICITY_PRICE_GBP_PER_KWH = 0.28

runtime_df["energy_kWh"] = runtime_df["runtime_hours"] * ASSUMED_POWER_KW
runtime_df["CO2e_g"] = runtime_df["energy_kWh"] * CARBON_INTENSITY_KG_PER_KWH * 1000
runtime_df["estimated_cost_gbp"] = runtime_df["energy_kWh"] * ELECTRICITY_PRICE_GBP_PER_KWH

runtime_df["notes"] = [
    "Local GeoTIFF loading, masking, and array preparation.",
    "Scaled K-means fitted on sampled pixels, then applied borough-wide.",
    "Spatial hold-out training, evaluation, and borough-wide prediction.",
]

total_row = pd.DataFrame(
    [
        {
            "stage": "Total",
            "runtime_seconds": runtime_df["runtime_seconds"].sum(),
            "runtime_minutes": runtime_df["runtime_minutes"].sum(),
            "runtime_hours": runtime_df["runtime_hours"].sum(),
            "energy_kWh": runtime_df["energy_kWh"].sum(),
            "CO2e_g": runtime_df["CO2e_g"].sum(),
            "estimated_cost_gbp": runtime_df["estimated_cost_gbp"].sum(),
            "notes": "Sum of measured notebook stages.",
        }
    ]
)

environmental_cost_df = pd.concat([runtime_df, total_row], ignore_index=True)

display_df = environmental_cost_df[
    [
        "stage",
        "runtime_hours",
        "energy_kWh",
        "CO2e_g",
        "estimated_cost_gbp",
        "notes",
    ]
].copy()

display_df = display_df.rename(
    columns={
        "stage": "Stage",
        "runtime_hours": "Runtime (hours)",
        "energy_kWh": "Energy (kWh)",
        "CO2e_g": "CO2e (g)",
        "estimated_cost_gbp": "Estimated cost (£)",
        "notes": "Notes",
    }
)
display_df.columns.name = None

display(display_df.round(5))

fig, ax = plt.subplots(figsize=(7.2, 3.6))
ax.bar(runtime_df["stage"], runtime_df["runtime_seconds"], color="#4c78a8")
ax.set_title("Measured runtime by workflow stage")
ax.set_ylabel("Runtime (seconds)")

plt.tight_layout()
fig.savefig(FIGURE_DIR / "environmental_cost.png", dpi=300, bbox_inches="tight")
plt.show()


## Context and mitigation

The estimated footprint is very small because the workflow relies on CPU-only classical machine learning, a borough-scale area of interest, and a limited set of modelling stages. In this run, pre-processed local GeoTIFF inputs also reduced the cost of data preparation relative to a full online compositing workflow.

- CPU-only K-means and Random Forest kept compute demand low.
- The Enfield borough boundary limited data volume while still supporting meaningful urban land-cover comparison.
- A 20 m working resolution provided a practical balance between spatial detail and computational efficiency.
- These values are simplified runtime-based estimates intended for workflow comparison rather than precise life-cycle accounting.


# References

Breiman, L. (2001) 'Random forests', *Machine Learning*, 45, pp. 5-32. https://doi.org/10.1023/A:1010933404324

ESA WorldCover Consortium (2021) *ESA WorldCover 10 m 2021 v200*. Available at: https://esa-worldcover.org/en (Accessed: 13 May 2026).

Google Earth Engine Data Catalog (n.d.) *Harmonized Sentinel-2 MSI: MultiSpectral Instrument, Level-2A (SR)*, `COPERNICUS/S2_SR_HARMONIZED`. Available at: https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED (Accessed: 13 May 2026).

Google Earth Engine Data Catalog (n.d.) *Sentinel-2: Cloud Probability*, `COPERNICUS/S2_CLOUD_PROBABILITY`. Available at: https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_CLOUD_PROBABILITY (Accessed: 13 May 2026).

Google Earth Engine Data Catalog (n.d.) *ESA WorldCover 10m v200*, `ESA/WorldCover/v200`. Available at: https://developers.google.com/earth-engine/datasets/catalog/ESA_WorldCover_v200 (Accessed: 13 May 2026).

Gorelick, N., Hancher, M., Dixon, M., Ilyushchenko, S., Thau, D. and Moore, R. (2017) 'Google Earth Engine: Planetary-scale geospatial analysis for everyone', *Remote Sensing of Environment*, 202, pp. 18-27. https://doi.org/10.1016/j.rse.2017.06.031

Greater London Authority (n.d.) *London Boroughs*. Available at: https://data.london.gov.uk/dataset/london_boroughs (Accessed: 13 May 2026).

MacQueen, J. (1967) 'Some methods for classification and analysis of multivariate observations', in Le Cam, L. M. and Neyman, J. (eds.) *Proceedings of the Fifth Berkeley Symposium on Mathematical Statistics and Probability, Volume 1: Statistics*. Berkeley, CA: University of California Press, pp. 281-297.

Pedregosa, F. et al. (2011) 'Scikit-learn: Machine Learning in Python', *Journal of Machine Learning Research*, 12, pp. 2825-2830. Available at: https://jmlr.org/papers/v12/pedregosa11a.html (Accessed: 13 May 2026).

Rouse, J. W., Haas, R. H., Schell, J. A. and Deering, D. W. (1974) 'Monitoring vegetation systems in the Great Plains with ERTS', in *Proceedings of the Third Earth Resources Technology Satellite-1 Symposium*. NASA SP-351, Vol. 1, pp. 309-317.

Zha, Y., Gao, J. and Ni, S. (2003) 'Use of normalized difference built-up index in automatically mapping urban areas from TM imagery', *International Journal of Remote Sensing*, 24(3), pp. 583-594. https://doi.org/10.1080/01431160304987


In [ ]:
requirements = """earthengine-api
geemap
geopandas
rasterio
numpy
pandas
matplotlib
scikit-learn
"""

requirements_path = PROJECT_DIR / "requirements.txt"
requirements_path.write_text(requirements, encoding="utf-8")

print(requirements_path)
print(requirements_path.read_text())


In [ ]:
readme_lines = [
    "# Evaluating Supervised and Unsupervised Land-Cover Classification in Urban Environments",
    "",
    "## A Sentinel-2 Case Study of Enfield, London",
    "",
    "This repository presents a GEOL0069 land-cover classification project comparing **K-means clustering** and **Random Forest classification** in the London Borough of Enfield. The analysis uses a **2021 April-August Sentinel-2 Level-2A seasonal composite**, Sentinel-2 cloud probability masking, and a remapped ESA WorldCover 2021 reference layer.",
    "",
    "The project is a **single-year classification comparison**, not a multi-year change detection study. Both methods use the same Sentinel-2 feature stack, so differences in the outputs can be interpreted as differences in modelling approach rather than differences in input imagery.",
    "",
    "## Research Question",
    "",
    "**How do supervised and unsupervised methods differ in their representation of borough-scale urban land cover in Enfield, and what does this reveal about interpretability, reference-label agreement, and computational cost?**",
    "",
    "The comparison uses four target classes:",
    "",
    "- Built-up / Urban",
    "- Vegetation",
    "- Water",
    "- Open land / Bare or sparsely vegetated surfaces",
    "",
    "## Project Motivation",
    "",
    "Urban land-cover classification is difficult because built surfaces, vegetation, water, bare ground, and open land often occur close together and can overlap spectrally within Sentinel-2 pixels. This is especially relevant in Enfield, where residential areas, parks, reservoirs, transport corridors, agricultural margins, and sparsely vegetated surfaces coexist within a compact borough-scale study area.",
    "",
    "The value of using Enfield is not simply that it contains diverse land-cover types. The important point is that this diversity creates a classification problem: class boundaries are spatially mixed, reference labels are imperfect, and different algorithms may respond differently to ambiguous urban surfaces. This makes Enfield a useful case study for evaluating the trade-off between unsupervised spectral structure and supervised semantic consistency.",
    "",
    "## Study Area",
    "",
    "The study area is the **London Borough of Enfield**, defined using an official London borough boundary layer. Enfield is located in north London and contains dense urban fabric, suburban residential areas, parks, reservoirs, river corridors, transport infrastructure, agricultural edges, and open or sparsely vegetated surfaces.",
    "",
    "At Sentinel-2 resolution, residential pixels may contain buildings, roads, gardens, street trees, and small green spaces. Similarly, open land and bare surfaces may overlap spectrally with dry grass, concrete, bright roofs, car parks, and rail corridors. This spatial and spectral mixture makes the borough suitable for comparing how K-means and Random Forest handle urban classification ambiguity.",
    "",
    "![Study area](figures/study_area.png)",
    "",
    "## Data Source and Pre-processing",
    "",
    "The workflow combines the Enfield boundary, Sentinel-2 Level-2A surface reflectance, Sentinel-2 cloud probability, and ESA WorldCover 2021 labels to produce an 8-feature Sentinel-2 array and a four-class reference label array for modelling.",
    "",
    "The analysis uses **April-August 2021** imagery. This leaf-on period was selected because vegetation is more spectrally distinct from built-up and bare surfaces, while avoiding unnecessary seasonal variation from winter conditions. A broad scene-level cloud filter (`CLOUDY_PIXEL_PERCENTAGE < 80`) retains enough summer observations, while pixel-level cloud masking is applied more strictly using Sentinel-2 cloud probability (`CLOUD_PROB_THRESHOLD = 40`).",
    "",
    "A **20 m** analysis scale was selected as a practical compromise between spatial detail and Colab runtime stability. In the local GeoTIFF fallback workflow, the feature and label rasters are checked after loading to confirm that they use the same projected coordinate reference system and pixel size. The rasters are stored in **EPSG:32630 (UTM Zone 30N)**, whose map units are metres; therefore, class-area estimates treat each pixel as **20 m x 20 m** before converting totals to square kilometres.",
    "",
    "![Sentinel-2 RGB composite](figures/sentinel2_rgb.png)",
    "",
    "### WorldCover Remapping",
    "",
    "| WorldCover class | Target class |",
    "|---|---|",
    "| Tree cover, shrubland, grassland, cropland, herbaceous wetland | Vegetation |",
    "| Built-up | Built-up / Urban |",
    "| Permanent water bodies | Water |",
    "| Bare / sparse vegetation, moss and lichen | Open land / Bare |",
    "| Snow and ice, mangroves | Ignored |",
    "",
    "## Method Overview",
    "",
    "Both classification methods use the same 2021 Sentinel-2 feature stack. The comparison therefore isolates the effect of the modelling approach: K-means is used as an unsupervised spectral baseline, while Random Forest is used as a supervised label-driven classifier trained with the remapped WorldCover reference raster.",
    "",
    "The feature stack contains six Sentinel-2 spectral bands and two indices:",
    "",
    "- B2, B3, B4",
    "- B8, B11, B12",
    "- NDVI",
    "- NDBI",
    "",
    "## K-means Workflow",
    "",
    "K-means is fitted to a scaled sample of valid Sentinel-2 pixels. The final model uses a pre-specified `k = 4` so that the unsupervised output can be translated into the same four target classes used by the Random Forest workflow. This choice is made for comparability, not because the diagnostics prove that Enfield naturally contains exactly four spectral groups.",
    "",
    "![K-means workflow](figures/kmeans_workflow.png)",
    "",
    "### K-means Diagnostics",
    "",
    "The elbow and silhouette plots are reported to make the pre-specified `k = 4` choice transparent. The final value of `k` is not selected automatically from these diagnostics; it is fixed to match the four target land-cover classes used throughout the comparison.",
    "",
    "![K-means diagnostics](figures/kmeans_diagnostics.png)",
    "",
    "### Heuristic Cluster Interpretation",
    "",
    "The K-means clusters are interpreted using cluster feature profiles and map inspection. These labels should be read as heuristic comparison labels rather than field-validated land-cover classes. In particular, the Built-up / Urban cluster represents mixed urban fabric at 20 m resolution rather than pure impervious surface.",
    "",
    "![K-means cluster profiles](figures/kmeans_cluster_profiles.png)",
    "",
    "## Random Forest Workflow",
    "",
    "Random Forest uses the same Sentinel-2 feature stack together with the remapped WorldCover reference raster. It is trained and evaluated with a block-based spatial hold-out split before being applied to all valid Enfield pixels.",
    "",
    "![Random Forest workflow](figures/rf_workflow.png)",
    "",
    "The supervised workflow provides clearer semantic class labels than K-means, but the evaluation should be interpreted as agreement with the WorldCover-derived reference layer rather than independent field-validated accuracy.",
    "",
    "## Results and Comparison",
    "",
    "The final maps show the common four-class scheme produced by the unsupervised and supervised workflows. K-means produces a more fragmented map because it responds directly to spectral structure. Random Forest produces a cleaner thematic map because it is constrained by the reference-label scheme.",
    "",
    "![Final classification maps](figures/final_classification_maps.png)",
    "",
    "### Random Forest Evaluation",
    "",
    "The Random Forest model achieved an overall agreement of **0.898** and a Cohen's kappa of **0.822** against the remapped WorldCover-derived labels under the block-based spatial hold-out design. These values indicate strong reference-label agreement, but they should not be interpreted as independent field-validated accuracy.",
    "",
    "The confusion matrix and feature-importance panel show that Vegetation and Water are more reliably identified, while Open land / Bare is weaker because it has fewer reference samples and overlaps spectrally with several urban materials.",
    "",
    "![Random Forest evaluation](figures/rf_evaluation.png)",
    "",
    "### Spatial Hold-out Design",
    "",
    "The spatial hold-out split is shown explicitly to make the evaluation design visible. This is more transparent and more conservative than a purely random pixel split, which could overestimate performance because neighbouring pixels often share similar spectral and land-cover properties.",
    "",
    "![Spatial hold-out design](figures/spatial_holdout.png)",
    "",
    "### Class Area Comparison",
    "",
    "Values are from the submitted reproducible run using `RANDOM_STATE = 42`.",
    "",
    "| Class | K-means (%) | Random Forest (%) | Difference, RF - K-means (pp) | Interpretation |",
    "|---|---:|---:|---:|---|",
    "| Built-up / Urban | 47.55 | 39.20 | -8.36 | RF maps less built-up land, reflecting closer alignment with the WorldCover reference-label scheme. |",
    "| Vegetation | 43.55 | 56.24 | +12.69 | RF maps more vegetation, likely influenced by gardens, parks, and mixed urban-green pixels. This is the largest method difference in the table. |",
    "| Water | 3.78 | 4.04 | +0.26 | Very similar between methods, consistent with strong water separability. |",
    "| Open land / Bare | 5.12 | 0.53 | -4.60 | K-means allocates more area to spectrally bright or low-vegetation surfaces, while RF maps much of this into other reference classes. |",
    "",
    "## Discussion and Limitations",
    "",
    "### Where K-means Adds Value",
    "",
    "K-means adds value by exposing spectral structure that is compressed once the analysis is translated into a fixed four-class land-cover scheme. In this project, `k = 4` is used to match the shared comparison framework rather than to claim that Enfield naturally contains only four spectral groups.",
    "",
    "The cluster profile plot and post-hoc interpretation table show that Enfield contains clearly vegetated and water-dominated surfaces, as well as more ambiguous combinations of built materials, sparse vegetation, and open ground. The Built-up / Urban cluster should be read carefully: its moderate NDVI indicates mixed urban fabric rather than pure impervious surface, which is plausible at 20 m resolution because residential pixels often contain buildings, roads, gardens, street trees, and small green spaces.",
    "",
    "K-means is therefore useful for spectral interpretation, but weaker as a final labelled land-cover map because its class names depend on post-hoc interpretation.",
    "",
    "### Where Random Forest Is More Reliable",
    "",
    "Random Forest produces the more submission-ready four-class land-cover map because it is tied to named classes and evaluated against the remapped WorldCover reference layer. Its strength is semantic clarity and consistency. However, the model is best understood as reproducing a reference classification scheme rather than independently verifying the true land cover of Enfield.",
    "",
    "### Ambiguous Surfaces in Enfield",
    "",
    "The largest differences between methods occur in the balance between Built-up / Urban, Vegetation, and Open land / Bare. Car parks, dry grass, bare soil, rail corridors, bright roofs, paved surfaces, and sparsely vegetated ground can overlap spectrally at 20 m resolution. By contrast, large reservoirs and dense vegetation are more distinct, which helps explain why Water and Vegetation are more stable across the two methods.",
    "",
    "The Open land / Bare class should be interpreted most cautiously. It has far fewer available samples than Built-up / Urban and Vegetation, and class weighting cannot fully compensate for limited and spectrally mixed reference samples. Its weaker F1-score is therefore likely caused by both class imbalance and genuine spectral overlap.",
    "",
    "### Implications for Borough-scale AI4EO Workflows",
    "",
    "The results show a trade-off between spectral flexibility and semantic consistency. K-means is useful for revealing internal variation and mixed urban surfaces, but it depends on post-hoc interpretation. Random Forest is more effective when the goal is a clearly labelled borough-scale product, but it is constrained by the quality, balance, and definitions of the reference labels used for training.",
    "",
    "Overall, the Enfield case study shows that the two methods answer different classification questions. Random Forest is better suited to producing a stable labelled land-cover product when a reference scheme is available, while K-means is more useful for exposing spectral ambiguity in mixed urban environments. The strongest interpretation therefore comes from reading the two outputs together rather than treating one method as a direct replacement for the other.",
    "",
    "## Environmental Cost",
    "",
    "The computational footprint of the workflow is estimated from measured stage runtimes. The estimate reflects the executed Colab run with local fallback inputs and does not include the upstream cost of generating the GeoTIFF data package.",
    "",
    "The workflow has a very small measured footprint because it uses CPU-only classical machine learning, a borough-scale area of interest, and a 20 m working resolution.",
    "",
    "![Environmental cost](figures/environmental_cost.png)",
    "",
    "## Repository Structure",
    "",
    "```text",
    "GEOL0069-23004877-Enfield-LandCover-Classification/",
    "|-- Enfield_LandCover_KMeans_RF_GEOL0069.ipynb",
    "|-- README.md",
    "|-- requirements.txt",
    "|-- data/",
    "|   |-- enfield_boundary_gla.geojson",
    "|   |-- enfield_s2_features_2021_20m.tif",
    "|   |-- enfield_worldcover4_2021_20m.tif",
    "|-- figures/",
    "|   |-- study_area.png",
    "|   |-- sentinel2_rgb.png",
    "|   |-- kmeans_workflow.png",
    "|   |-- kmeans_diagnostics.png",
    "|   |-- kmeans_cluster_profiles.png",
    "|   |-- rf_workflow.png",
    "|   |-- final_classification_maps.png",
    "|   |-- rf_evaluation.png",
    "|   |-- spatial_holdout.png",
    "|   |-- environmental_cost.png",
    "```",
    "",
    "## Reproducibility",
    "",
    "The notebook can be opened in Google Colab or Jupyter and run from top to bottom. If Earth Engine authentication is unavailable, keep `USE_EE = False` and use the local GeoTIFF fallback files in the `data/` folder.",
    "",
    "Install dependencies with:",
    "",
    "```bash",
    "pip install -r requirements.txt",
    "```",
    "",
    "## References",
    "",
    "Breiman, L. (2001) 'Random forests', *Machine Learning*, 45, pp. 5-32. https://doi.org/10.1023/A:1010933404324",
    "",
    "ESA WorldCover Consortium (2021) *ESA WorldCover 10 m 2021 v200*. Available at: https://esa-worldcover.org/en (Accessed: 13 May 2026).",
    "",
    "Google Earth Engine Data Catalog (n.d.) *Harmonized Sentinel-2 MSI: MultiSpectral Instrument, Level-2A (SR)*, `COPERNICUS/S2_SR_HARMONIZED`. Available at: https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED (Accessed: 13 May 2026).",
    "",
    "Google Earth Engine Data Catalog (n.d.) *Sentinel-2: Cloud Probability*, `COPERNICUS/S2_CLOUD_PROBABILITY`. Available at: https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_CLOUD_PROBABILITY (Accessed: 13 May 2026).",
    "",
    "Google Earth Engine Data Catalog (n.d.) *ESA WorldCover 10m v200*, `ESA/WorldCover/v200`. Available at: https://developers.google.com/earth-engine/datasets/catalog/ESA_WorldCover_v200 (Accessed: 13 May 2026).",
    "",
    "Gorelick, N., Hancher, M., Dixon, M., Ilyushchenko, S., Thau, D. and Moore, R. (2017) 'Google Earth Engine: Planetary-scale geospatial analysis for everyone', *Remote Sensing of Environment*, 202, pp. 18-27. https://doi.org/10.1016/j.rse.2017.06.031",
    "",
    "Greater London Authority (n.d.) *London Boroughs*. Available at: https://data.london.gov.uk/dataset/london_boroughs (Accessed: 13 May 2026).",
    "",
    "MacQueen, J. (1967) 'Some methods for classification and analysis of multivariate observations', in Le Cam, L. M. and Neyman, J. (eds.) *Proceedings of the Fifth Berkeley Symposium on Mathematical Statistics and Probability, Volume 1: Statistics*. Berkeley, CA: University of California Press, pp. 281-297.",
    "",
    "Pedregosa, F. et al. (2011) 'Scikit-learn: Machine Learning in Python', *Journal of Machine Learning Research*, 12, pp. 2825-2830. Available at: https://jmlr.org/papers/v12/pedregosa11a.html (Accessed: 13 May 2026).",
    "",
    "Rouse, J. W., Haas, R. H., Schell, J. A. and Deering, D. W. (1974) 'Monitoring vegetation systems in the Great Plains with ERTS', in *Proceedings of the Third Earth Resources Technology Satellite-1 Symposium*. NASA SP-351, Vol. 1, pp. 309-317.",
    "",
    "Zha, Y., Gao, J. and Ni, S. (2003) 'Use of normalized difference built-up index in automatically mapping urban areas from TM imagery', *International Journal of Remote Sensing*, 24(3), pp. 583-594. https://doi.org/10.1080/01431160304987",
]

readme = "\n".join(readme_lines)

readme_path = PROJECT_DIR / "README.md"
readme_path.write_text(readme, encoding="utf-8")

print(readme_path)
print("Final README.md created successfully.")


In [ ]:
from shutil import copy2

NOTEBOOK_SOURCE = Path("/content/drive/MyDrive/Enfield_LandCover_KMeans_RF_GEOL0069.ipynb")
NOTEBOOK_TARGET = PROJECT_DIR / "Enfield_LandCover_KMeans_RF_GEOL0069.ipynb"

copy2(NOTEBOOK_SOURCE, NOTEBOOK_TARGET)

print(NOTEBOOK_TARGET)
print("Notebook copied successfully.")




In [ ]:
print(f"Project folder: {PROJECT_DIR}")
print("\nProject contents:")

for path in sorted(PROJECT_DIR.rglob("*")):
    relative_path = path.relative_to(PROJECT_DIR)
    if path.is_file():
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f"- {relative_path} ({size_mb:.2f} MB)")
    else:
        print(f"- {relative_path}/")


In [ ]:
import shutil

zip_base = Path("/content/GEOL0069-Enfield-LandCover-Classification")
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=PROJECT_DIR)

print(zip_path)
print("Project zip created successfully.")
